<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: The forward pass

**[The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/)** · Deep Learning and Transformers Explained · Module 2, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn Larkfield's tickets into a bag of words, send a batch through an untrained network, check the shape after every layer, and fix a shape mismatch from its message.

| | |
|---|---|
| **Time** | About 40 minutes, with the module practice |
| **Needs** | A browser and a free Colab or Kaggle account. The setup cells download the tickets (about 2 MB) and the course's `ticketnet.py`. In Kaggle, turn on Internet in the notebook settings. |
| **You should know** | Shapes, the batch dimension and broadcasting, from [Tensors and shapes](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/); the seed, from [Set up PyTorch](https://learning.nextia-ai.com/courses/deep-learning/m02/set-up-pytorch/); layers and ReLU, from Module 1. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M02-L03-the-forward-pass/the-forward-pass-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Tickets have different lengths

A layer reads the same number of inputs for every ticket. Tickets have different lengths. `tokenize()` cuts a text into **tokens**: lower-case words, numbers and single punctuation marks.

> **Predict.** The three tiny tickets have 4, 4 and 7 tokens. Can `torch.tensor()` make one tensor from their lists of token positions? Run the two cells.

In [ ]:
import re

def tokenize(text):
    """Lower-case words, numbers and single punctuation marks."""
    return re.findall(r"[a-z]+|[0-9]+|[^\sa-z0-9]", text.lower())

tiny = ["Parcel not arrived.", "Card charged twice.", "Parcel arrived broken, broken lamp."]
for text in tiny:
    print(len(tokenize(text)), tokenize(text))

In [ ]:
tiny_vocab = sorted({token for text in tiny for token in tokenize(text)})
ids = [[tiny_vocab.index(token) for token in tokenize(text)] for text in tiny]
print(ids)
try:
    print(torch.tensor(ids))
except ValueError as error:  # print the message, so the notebook can go on
    print("ValueError:", error)

> **Check.** You should see `ValueError: expected sequence of length 4 at dim 1 (got 7)`. Rows of different lengths do not make a box.

A **bag of words** gives every ticket one count for each entry of a fixed **vocabulary**. Run the cell.

In [ ]:
print(tiny_vocab)

def counts(text, vocab):
    """One number per vocabulary entry: how often it appears in the text."""
    x = torch.zeros(len(vocab))
    for token in tokenize(text):
        x[vocab.index(token)] += 1
    return x

print(torch.stack([counts(text, tiny_vocab) for text in tiny]))

> **Check.** You should see 10 vocabulary entries and a (3, 10) tensor. The third row has a `2.` for "broken".

## 2. The vocabulary and the bag of words of the tickets

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them. The vocabulary comes from the training tickets only, with every token that appears at least twice, plus `<pad>` (position 0) and `<unk>` (position 1, for any other token).

> **Predict.** The training tickets have 4,964 different tokens. Will the vocabulary be larger or smaller? Run the cell.

In [ ]:
from collections import Counter

word_counts = Counter(token for row in train_rows for token in tokenize(row["text"]))
itos = ["<pad>", "<unk>"] + sorted(t for t, c in word_counts.items() if c >= 2)   # position -> token
stoi = {token: i for i, token in enumerate(itos)}                                 # token -> position
print(len(word_counts), "different tokens in train,", len(itos), "in the vocabulary")

> **Check.** You should see `4964 different tokens in train, 2864 in the vocabulary`.

`bag()` makes the counts of one ticket, from its first 120 tokens, as `ticketnet.py` does. Run the cell.

In [ ]:
def bag(text):
    """The counts of one ticket: shape (vocabulary size,)."""
    x = torch.zeros(len(itos))
    for token in tokenize(text)[:120]:     # the first 120 tokens, as in ticketnet.py
        x[stoi.get(token, 1)] += 1          # 1 is <unk>: a token not in the vocabulary
    return x

ticket = valid_rows[0]
x = bag(ticket["text"])
print(ticket["ticket_id"], ticket["team"], "|", len(tokenize(ticket["text"])), "tokens")
print("shape", tuple(x.shape), "| non-zero entries", int((x > 0).sum()), "| sum", int(x.sum()))
x8 = torch.stack([bag(row["text"]) for row in valid_rows[:8]])
print(x8.shape)

> **Check.** You should see `T-64697 payment | 92 tokens`, `shape (2864,) | non-zero entries 70 | sum 92` and `torch.Size([8, 2864])`.

## 3. The network, shape by shape

`nn.Linear(2864, 64)` is a layer of 64 neurons that each read all 2,864 counts. `nn.ReLU()` sets negatives to 0. `nn.Linear(64, 5)` gives 5 scores, one for each team. Seed 0 makes the random starting weights the same as in the lesson.

> **Predict.** Write down the shape after each step: `log1p`, the first Linear, ReLU and the last Linear. Then run the cell.

In [ ]:
from torch import nn

TEAMS = ["delivery", "returns", "payment", "warranty", "account"]
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(len(itos), 64),   # 2,864 counts in, 64 numbers out
    nn.ReLU(),
    nn.Linear(64, len(TEAMS)),  # 64 numbers in, 5 scores out
)
first, relu, last = model
h = torch.log1p(x8);  print("input     ", tuple(h.shape))
h = first(h);         print("Linear    ", tuple(h.shape))
h = relu(h);          print("ReLU      ", tuple(h.shape))
scores = last(h);     print("Linear    ", tuple(scores.shape))

> **Check.** You should see (8, 2864), (8, 64), (8, 64) and (8, 5). The batch, 8, never changes.

> **Your turn.** Compute `n_parameters`: the number of parameters of `model`. `p.numel()` counts the numbers of one parameter `p`, and `model.parameters()` gives every parameter. Then run the check cell.

In [ ]:
n_parameters = sum(p.numel() for p in model.parameters())
print(n_parameters)

In [ ]:
expect('n_parameters', n_parameters, 183685)

## 4. What the untrained network says

> **Predict.** The weights are random. How often will the network choose the right team? Run the cell.

In [ ]:
print(scores[0])
x_valid = torch.stack([bag(row["text"]) for row in valid_rows])
y_valid = torch.tensor([TEAMS.index(row["team"]) for row in valid_rows])
with torch.no_grad():
    pred = model(torch.log1p(x_valid)).argmax(dim=1)
print("accuracy", round((pred == y_valid).float().mean().item(), 3))
print("predicted:", Counter(TEAMS[i] for i in pred.tolist()).most_common())

> **Check.** You should see the scores `-0.1288, 0.0228, 0.1072, 0.0242, 0.0647`, `accuracy 0.182` and `account` 438 times. That is chance level: the weights are random, and they favour account.

## 5. The same code in ticketnet.py

The course's `ticketnet.py` holds the same code: `Vocab`, `bag_of_words()` and `BagOfWordsNet`. Run the cell.

In [ ]:
import ticketnet

vocab = ticketnet.Vocab([row["text"] for row in train_rows])
print(len(vocab), vocab.itos == itos)
ticketnet.set_seed(0)
net = ticketnet.BagOfWordsNet(len(vocab))
x_same = ticketnet.bag_of_words(valid_rows[:8], vocab)
print(torch.equal(x_same, x8), torch.allclose(net(x_same), scores))

> **Check.** You should see `2864 True` and `True True`: the same vocabulary, counts and scores.

## 6. Failure case: "PyTorch will figure out the shapes"

Tomás builds a vocabulary with every token, also the rare ones, and sends its counts to `model`.

> **Predict.** What happens? Run the cell.

In [ ]:
itos_all = ["<pad>", "<unk>"] + sorted(word_counts)          # every token, also the rare ones
stoi_all = {token: i for i, token in enumerate(itos_all)}
x8_all = torch.zeros(8, len(itos_all))
for row_i, row in enumerate(valid_rows[:8]):
    for token in tokenize(row["text"])[:120]:
        x8_all[row_i, stoi_all.get(token, 1)] += 1
print(x8_all.shape)
try:
    print(model(torch.log1p(x8_all)).shape)
except RuntimeError as error:  # print the message, so the notebook can go on
    print("RuntimeError:", error)

> **Check.** You should see `torch.Size([8, 4966])` and `mat1 and mat2 shapes cannot be multiplied (8x4966 and 2864x64)`. The first layer was built for 2,864 counts.

The fix builds the first layer from the vocabulary, never from a typed number. Run the cell.

In [ ]:
torch.manual_seed(0)
model_all = nn.Sequential(nn.Linear(len(itos_all), 64), nn.ReLU(), nn.Linear(64, len(TEAMS)))
print(model_all(torch.log1p(x8_all)).shape)

> **Check.** You should see `torch.Size([8, 5])`.

## 7. Change one thing: a wider hidden layer

> **Your turn.** Make `wide`: the same network with 128 hidden neurons instead of 64, with seed 0. Predict the shape after its first layer, its output shape and its number of parameters first. Then put the number of parameters in `wide_parameters`, and run the check cell.

In [ ]:
torch.manual_seed(0)
wide = nn.Sequential(nn.Linear(len(itos), 128), nn.ReLU(), nn.Linear(128, len(TEAMS)))
print(wide(torch.log1p(x8)).shape, wide[0](torch.log1p(x8)).shape)
wide_parameters = sum(p.numel() for p in wide.parameters())
print(wide_parameters)

In [ ]:
expect('wide_parameters', wide_parameters, 367365)

## 8. Module practice: a tiny network

The worked example sends Priya's three tickets through a network with 4 inputs, 3 hidden neurons and 2 outputs, and prints the shape and the first ticket's numbers after each layer. Run the cell.

In [ ]:
batch = torch.tensor([[2.0, 2.0, 0.0, 1.0],
                      [1.0, 0.0, 3.0, 0.0],
                      [3.0, 1.0, 1.0, 1.0]])
torch.manual_seed(0)
tiny_net = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2))
h = batch
for layer in tiny_net:
    h = layer(h)
    print(f"{layer.__class__.__name__:<6} {tuple(h.shape)}  {h[0].detach().numpy().round(3)}")

> **Check.** You should see (3, 3), (3, 3) and (3, 2). After ReLU, the first ticket's numbers are all 0, because all three were negative.

> **Your turn.** Make `task_net` with 4 inputs, **6** hidden neurons and 2 outputs, with seed 0. Send `five = torch.ones(5, 4)` through it, one layer at a time. Put the three shapes, as tuples, in `task_shapes`, and the number of parameters in `task_parameters`. Predict both first. Then run the check cells.

In [ ]:
five = torch.ones(5, 4)
torch.manual_seed(0)
task_net = nn.Sequential(nn.Linear(4, 6), nn.ReLU(), nn.Linear(6, 2))
h = five
task_shapes = []
for layer in task_net:
    h = layer(h)
    task_shapes.append(tuple(h.shape))
task_parameters = sum(p.numel() for p in task_net.parameters())
print(task_shapes, task_parameters)

In [ ]:
expect('task_shapes', task_shapes, [(5, 6), (5, 6), (5, 2)])

In [ ]:
expect('task_parameters', task_parameters, 44)

## 9. Module check: fix a shape mismatch

> **Your turn.** This network stops with a shape error. Run the cell and read the message. Then write `fixed`: the same network, with the mistake corrected, and run the check cell.

In [ ]:
broken = nn.Sequential(nn.Linear(4, 6), nn.ReLU(), nn.Linear(5, 2))
try:
    print(broken(five).shape)
except RuntimeError as error:  # print the message, so the notebook can go on
    print("RuntimeError:", error)

fixed = nn.Sequential(nn.Linear(4, 6), nn.ReLU(), nn.Linear(6, 2))
print(fixed(five).shape)

In [ ]:
expect('the output shape of fixed', tuple(fixed(five).shape) if isinstance(fixed, nn.Module) else fixed, (5, 2))

## Next

You can turn tickets into a bag of words, read the shape after every layer, and fix a shape error from its message. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/) has the layer diagram, the explorer of shape errors and the module check that counts toward your certificate. Next, in Module 3, you measure how wrong the scores are and start to train the network.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Loss: how wrong is the network?](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.